In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
print("libraries imported successfully.")

In [ ]:
matches=pd.read_csv("../data/processed/epl_matches_stage1.csv")
matches["Date"]=matches["Date"].astype("string").str.replace(r"\s*\(P\)$", "", regex=True)
matches["Date"]=pd.to_datetime(
    matches["Date"],
    errors="coerce",
    dayfirst=True
)
print(matches.tail())
matches.shape

In [ ]:
#Investigate data quality
missing_report=pd.DataFrame({
    "Missing_Count":matches.isnull().sum(),
    "Missing_Percentage": (matches.isnull().mean()*100).round(2)
})
print(missing_report)
#Duplicate Records
duplicate_count=matches.duplicated(subset=["Date", "HomeTeam", "AwayTeam", "FullTimeScore"]).sum()
print("Potential duplicate matches: ", duplicate_count)
#Invalid Scores
invalid_scores=matches[
    matches["HomeGoals"].isna()|matches["AwayGoals"].isna()
]
print("Invalid score records: ", len(invalid_scores))

In [ ]:
#Cleaning
clean_matches=matches.copy()
#teamnames
clean_matches["HomeTeam"]=clean_matches["HomeTeam"].astype("string").str.strip()
clean_matches["AwayTeam"]=clean_matches["AwayTeam"].astype("string").str.strip()
#Remove invalid essential records
clean_matches = clean_matches.dropna(
    subset=[
        "Date",
        "HomeTeam",
        "AwayTeam",
        "HomeGoals",
        "AwayGoals"
    ]
).copy()
#Goal columns to integer
clean_matches["HomeGoals"] = (
    clean_matches["HomeGoals"].astype(int)
)
clean_matches["AwayGoals"] = (
    clean_matches["AwayGoals"].astype(int)
)

#Remove exact duplicate matches
clean_matches = clean_matches.drop_duplicates(
    subset=[
        "Date",
        "HomeTeam",
        "AwayTeam",
        "FullTimeScore"
    ],
    keep="first"
).copy()
print(clean_matches)

In [ ]:
#Match result
conditions = [
    clean_matches["HomeGoals"] > clean_matches["AwayGoals"],
    clean_matches["HomeGoals"] < clean_matches["AwayGoals"],
    clean_matches["HomeGoals"] == clean_matches["AwayGoals"]
]

clean_matches["MatchResult"] = np.select(
    conditions,
    ["HW", "AW", "D"],
    default="Unknown"
)
clean_matches=(clean_matches.sort_values("Date").reset_index(drop=True))
print(clean_matches)

In [ ]:
#Adding columns
clean_matches["TotalGoals"]=(clean_matches["HomeGoals"]+clean_matches["AwayGoals"])
clean_matches["GoalDifference"]=(clean_matches["HomeGoals"]-clean_matches["AwayGoals"])
clean_matches["Year"]=(clean_matches["Date"].dt.year)
clean_matches["Month"]=(clean_matches["Date"].dt.month)
print(clean_matches.shape)

In [ ]:
# EDA
#Match outcome distribution
result_counts=(clean_matches["MatchResult"].value_counts().reindex(["HW", "AW", "D"], fill_value=0))
result_percentage=(result_counts/result_counts.sum()*100).round(2)
print(result_counts)
print(result_percentage)
plt.figure(figsize=(10, 4))
sns.barplot(x=result_counts.index, y=result_counts.values)
plt.title("Distribution of Match Results")
plt.xlabel("Match Result")
plt.ylabel("Number of Matches")
plt.show()

In [ ]:
#Home advantage analysis
home_average=clean_matches["HomeGoals"].mean()
away_average= clean_matches["AwayGoals"].mean()
print("Average Home Goals :", round(home_average, 2))
print("Average Away Goals: ", round(away_average, 2))
goal_comparision=pd.DataFrame({
    "Team Type": ["Home", "Away"],
    "Average Goals": [home_average, away_average]
})
plt.figure(figsize=(10,4))
sns.barplot(data=goal_comparision, x="Team Type", y="Average Goals")
plt.title("Average Goals: Home vs Away")
plt.ylabel("Average Goals per match")
plt.show()

In [ ]:
#Goal Distribution
plt.figure(figsize=(9, 5))
sns.histplot(data=clean_matches, x="TotalGoals", bins=15)
plt.title("Distribution of total match goals")
plt.xlabel("Total Goals")
plt.ylabel("Number of matches")
plt.show()

#Seasonal
season_goals=(clean_matches.groupby("Source_File").agg(
    Matches=("TotalGoals", "count"),
    Average_Goals=("TotalGoals", "mean"),
    Home_Goals=("HomeGoals", "mean"),
    Away_Goals=("AwayGoals", "mean")
).round(2).sort_index())
print(season_goals)
plt.figure(figsize=(12, 5))
sns.lineplot(
    data=season_goals,
    x=season_goals.index,
    y="Average_Goals",
    marker="o"
)
plt.title("Average Goals Per Season")
plt.xlabel("Season")
plt.ylabel("Average Goals per Match")
plt.xticks(rotation=45)

plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#Top scoring teams
home_team_goals=(clean_matches.groupby("HomeTeam")["HomeGoals"].sum().sort_values(ascending=False))
print(home_team_goals.head())
away_team_goals=(clean_matches.groupby("AwayTeam")["AwayGoals"].sum().sort_values(ascending=False))
print(away_team_goals.head())
plt.figure(figsize=(9, 5))
sns.barplot(x=home_team_goals.values, y=home_team_goals.index)
plt.title("Top 10 teams by total home goals")
plt.xlabel("Total Home Goals")
plt.ylabel("Team")
plt.show()
plt.figure(figsize=(9, 5))
sns.barplot(x=away_team_goals.values, y=away_team_goals.index)
plt.title("Top 10 teams by total away goals")
plt.xlabel("Total Away Goals")
plt.ylabel("Team")
plt.show()

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
clean_matches.to_csv("../data/processed/epl_matches_clean.csv", index=False)
print("Cleaned Dataset saved successfully.")

## OBESERVATION:
~~~
i. There is no invalid score records.
ii. MatchResult: Home Wins: 1876 matches, Away Wins: 1276 matches, Draw: 1016 matches
iii. Top Scoring Home Teams: 
Manchester City FC      496
Liverpool FC            407
Arsenal FC              398
Chelsea FC              386
Manchester United FC    380
iv: Top Scoring Away Teams:
Manchester City FC      362
Liverpool FC            322
Tottenham Hotspur FC    306
Chelsea FC              305
Arsenal FC              304
